# Vegetation Response (NDVI) in North Turkana, Kenya

Use 2015-2022 CHIRPS rainfall and quality-screened MODIS NDVI to compare correlations at lags of 0-4 months, before and after removing the seasonal cycle. Read the student guide for site selection and interpretation. Run each cell in order.

In Colab, choose File -> Save a copy in Drive, then run the cells in order. The last cell saves your results in your Google Drive.

## 1 Set up and connect
Run this cell first in every new session. It installs and imports the required libraries, then prompts for your Earth Engine project ID and sign-in. Complete the prompts before moving on. Run the following cells in order and pause to draw or upload your boundary when prompted.


In [ ]:
# Install the notebook dependencies before importing them.
import subprocess
import sys
subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install',
                       'earthengine-api', 'geemap', 'geopandas', 'pyogrio'])

import hashlib
import json
import tempfile
import time
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import ee
import geemap
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm
from google.colab import drive, files, output as colab_output

colab_output.enable_custom_widget_manager()
project_id = input('Enter your Earth Engine Cloud project ID: ').strip()
if not project_id:
    raise ValueError('Enter the project ID from your Earth Engine registration.')
ee.Authenticate()
ee.Initialize(project=project_id)
print(ee.String('Connected to Earth Engine').getInfo())


## 2 Load the products
The next cells let you draw a rectangle or polygon, upload a zipped shapefile, or enter rectangle coordinates. Both rainfall and NDVI will be averaged over your selected area, at their native grid scales.

In [ ]:
chirps = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY').select('precipitation')
modis = ee.ImageCollection('MODIS/061/MOD13Q1')
rain_grid = chirps.first().projection()
ndvi_grid = modis.first().select('NDVI').projection()

def screen_ndvi(image):
    good = image.select('SummaryQA').eq(0)
    land = image.select('DetailedQA').rightShift(11).bitwiseAnd(7).eq(1)
    return (image.select('NDVI').multiply(0.0001)
            .updateMask(good.And(land))
            .copyProperties(image, ['system:time_start']))

ndvi = modis.map(screen_ndvi)

## 3 Choose your boundary

Run the map cell and then choose one method from the **boundary_method** dropdown in the following form.

- **Draw on map** — draw a rectangle or polygon, then run the form cell.
- **Upload shapefile or GeoJSON** — choose a ZIP with matching .shp, .shx, .dbf and .prj files or one GeoJSON file when prompted.
- **Enter rectangle coordinates** — enter west, south, east and north in decimal degrees when prompted.

After changing your boundary, rerun monthly extraction, lag analysis and saving. Both datasets are averaged over the chosen polygon.


In [ ]:
study_map = geemap.Map(center=[4.2, 35.7], zoom=8, basemap='Esri.WorldImagery')
display(study_map)


In [ ]:
# Choose a boundary method in the form, then run this cell.
boundary_method = "Draw on map" #@param ["Draw on map", "Upload shapefile or GeoJSON", "Enter rectangle coordinates"]
selection = {"Draw on map": "draw", "Upload shapefile or GeoJSON": "upload", "Enter rectangle coordinates": "coordinates"}[boundary_method]
region = None
if selection == 'draw':
    feature = study_map.draw_last_feature
    if feature is None:
        raise ValueError('Draw a rectangle or polygon on the map, then rerun this cell.')
    region = ee.Feature(feature).geometry()
elif selection == 'coordinates':
    west, south, east, north = map(float, input('West, south, east, north separated by commas: ').split(','))
    if not (-180 <= west < east <= 180 and -90 <= south < north <= 90):
        raise ValueError('Check the coordinate order and ranges.')
    region = ee.Geometry.Rectangle([west, south, east, north])
elif selection == 'upload':
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Upload one zipped shapefile or one GeoJSON file.')
    name, content = next(iter(uploaded.items()))
    suffix = Path(name).suffix.lower()
    with tempfile.TemporaryDirectory() as folder:
        source = Path(folder) / Path(name).name
        source.write_bytes(content)
        if suffix == '.zip':
            with zipfile.ZipFile(source) as archive:
                shapes = [item for item in archive.namelist() if item.lower().endswith('.shp')]
                if len(shapes) != 1:
                    raise ValueError('The ZIP must contain exactly one shapefile.')
                stem = shapes[0][:-4].lower()
                members = {item.lower() for item in archive.namelist()}
                if not all(stem + extension in members for extension in ['.shp', '.shx', '.dbf', '.prj']):
                    raise ValueError('Include .shp, .shx, .dbf and .prj with the same base name.')
            # Read the selected shapefile explicitly, including nested folders.
            # Copy its companion files into a private temporary directory.
            shape_name = shapes[0]
            selected_stem = shape_name[:-4].lower()
            extracted = Path(folder) / 'boundary'
            extracted.mkdir()
            with zipfile.ZipFile(source) as archive:
                for member in archive.namelist():
                    member_path = Path(member)
                    if member[:-len(member_path.suffix)].lower() == selected_stem:
                        extension = member_path.suffix.lower()
                        if extension in ['.shp', '.shx', '.dbf', '.prj', '.cpg']:
                            (extracted / ('boundary' + extension)).write_bytes(archive.read(member))
            boundaries = gpd.read_file(extracted / 'boundary.shp')
        elif suffix in ['.geojson', '.json']:
            boundaries = gpd.read_file(source)
        else:
            raise ValueError('Use a ZIP shapefile or GeoJSON.')
    if boundaries.empty or boundaries.crs is None:
        raise ValueError('The boundary must contain features and a defined coordinate reference system.')
    if not boundaries.geometry.geom_type.isin(['Polygon', 'MultiPolygon']).all():
        raise ValueError('Use polygon boundaries.')
    if not boundaries.geometry.is_valid.all():
        raise ValueError('Repair invalid polygons in GIS before uploading.')
    boundaries = boundaries.to_crs('EPSG:4326')
    region = geemap.geopandas_to_ee(boundaries).geometry()
else:
    raise ValueError('Choose draw, upload or coordinates.')

if region.type().getInfo() not in ['Polygon', 'MultiPolygon']:
    raise ValueError('Choose a rectangle or polygon.')
area_km2 = region.area(10).divide(1e6).getInfo()
print(f'Selected area: {area_km2:.1f} square kilometres')
study_map.addLayer(region, {'color': 'yellow'}, 'Selected boundary')
study_map.centerObject(region)
map_month = ee.Date('2020-04-01')
image = ndvi.filterDate(map_month, map_month.advance(1, 'month')).mean().clip(region)
study_map.addLayer(image, {'min': 0, 'max': 0.5, 'palette': ['f5e8c7', '78b56c', '155d27']}, 'April 2020 NDVI')
print('Inspect the selected outline and land cover on the map before continuing.')


## 4 Build the monthly table
Sum daily rainfall to mm per month. Average valid NDVI composites by timestamp month, then average their pixels in the buffer. Inspect missing values and preserve their calendar rows.

In [ ]:
start = ee.Date('2015-01-01')

def monthly_row(offset):
    begin = start.advance(offset, 'month')
    end = begin.advance(1, 'month')
    rain = chirps.filterDate(begin, end).sum().rename('rain_mm')
    green = ndvi.filterDate(begin, end).mean().rename('ndvi')
    rainfall = rain.reduceRegion(ee.Reducer.mean(), region, crs=rain_grid, maxPixels=100000000)
    greenness = green.reduceRegion(ee.Reducer.mean(), region, crs=ndvi_grid, maxPixels=100000000)
    return ee.Feature(None, rainfall.combine(greenness)).set('month', begin.format('YYYY-MM'))

# Calculate one month at a time to limit concurrent work.
# Completed months are retained in this runtime if you rerun the cell.

cache_definition = {'boundary': region.getInfo(), 'period': '2015-2022',
                    'ndvi': 'MOD13Q1.061 good land', 'rain': 'CHIRPS v2 mean'}
cache_key = hashlib.sha256(json.dumps(cache_definition, sort_keys=True).encode()).hexdigest()
if globals().get('_monthly_cache_key') != cache_key:
    _monthly_cache = {}
    _monthly_cache_key = cache_key

for offset in tqdm(range(96), desc='Monthly rainfall and NDVI'):
    if offset in _monthly_cache:
        continue
    for attempt in range(3):
        try:
            _monthly_cache[offset] = monthly_row(offset).getInfo()['properties']
            break
        except ee.EEException as error:
            message = str(error).lower()
            if attempt == 2 or not any(text in message for text in ['429', 'too many requests', 'too many concurrent']):
                raise
            pause = 30 * (attempt + 1)
            print(f'Earth Engine is busy. Waiting {pause} seconds before retrying this month.')
            time.sleep(pause)
    time.sleep(1)
records = [_monthly_cache[offset] for offset in range(96)]
data = pd.DataFrame(records).reindex(columns=['month', 'rain_mm', 'ndvi'])
data = data.sort_values('month').reset_index(drop=True)
data[['rain_mm', 'ndvi']] = data[['rain_mm', 'ndvi']].apply(pd.to_numeric, errors='coerce')
if data['ndvi'].notna().sum() < 2:
    raise ValueError('Too few valid NDVI months. Inspect the boundary for water or missing land observations, then select rangeland and rerun extraction.')
expected = pd.period_range('2015-01', '2022-12', freq='M').astype(str).tolist()
assert data['month'].tolist() == expected, 'Check the monthly calendar.'
print(data.isna().sum())
display(data.head())
time_axes = data.set_index(pd.to_datetime(data['month']))[['rain_mm', 'ndvi']].plot(subplots=True, figsize=(10, 5))
time_figure = time_axes[0].figure
plt.tight_layout()
plt.show()

## 5 Compare raw and seasonal anomaly correlations
A positive lag pairs earlier rain with later NDVI. For example, January rainfall pairs with February NDVI at lag 1. The anomaly subtracts the average for that calendar month. Compare peak shape and valid pair counts.

In [ ]:
calendar_month = pd.to_datetime(data['month']).dt.month
series = data[['rain_mm', 'ndvi']]
anomalies = series - series.groupby(calendar_month).transform('mean')

def correlation_at_lag(values, lag):
    paired = pd.concat([values['rain_mm'], values['ndvi'].shift(-lag)], axis=1).dropna()
    correlation = paired['rain_mm'].corr(paired['ndvi']) if len(paired) >= 2 else float('nan')
    return correlation, len(paired)

results = pd.DataFrame([
    (lag, *correlation_at_lag(series, lag), *correlation_at_lag(anomalies, lag))
    for lag in range(5)
], columns=['lag_months', 'raw_r', 'raw_pairs', 'anomaly_r', 'anomaly_pairs'])
display(results)
lag_axis = results.plot(x='lag_months', y=['raw_r', 'anomaly_r'], marker='o', xticks=range(5))
lag_axis.set_ylabel('Pearson correlation')
lag_figure = lag_axis.figure
plt.show()

## 6 Save the results to Google Drive
Run this cell and grant Colab access to your own Drive. Each run gets its own folder. Open My Drive -> Turkana NDVI Course to find the two CSV files and two plots. Report whether a clear positive peak persists after removing seasonality and include valid pair counts.

In [ ]:
drive.mount('/content/drive')
my_drive = Path('/content/drive/MyDrive')
if not my_drive.is_dir():
    raise RuntimeError('Google Drive is unavailable. Reconnect it and rerun this cell.')

run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
output = my_drive / 'Turkana NDVI Course' / run_id
output.mkdir(parents=True, exist_ok=False)
(output / 'study_boundary.geojson').write_text(json.dumps(ee.Feature(region).getInfo()), encoding='utf-8')
data.to_csv(output / 'turkana_rain_ndvi_monthly.csv', index=False)
results.to_csv(output / 'turkana_lag_results.csv', index=False)
time_figure.savefig(output / 'monthly_rain_ndvi.png', dpi=180, bbox_inches='tight')
lag_figure.savefig(output / 'lag_comparison.png', dpi=180, bbox_inches='tight')
print(f'Saved to My Drive / Turkana NDVI Course / {run_id}')
for file in sorted(output.iterdir()):
    print(file.name)
